In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from codecarbon import EmissionsTracker

In [2]:
# --- Load and prep ---
df = pd.read_csv("HRDataset_cleaned.csv")
df['Age'] = df['Age'].abs()
leakage_cols = [col for col in df.columns if col.startswith('EmploymentStatus_')]
df = df.drop(columns=leakage_cols + ['State', 'EmpStatusID', 'Tenure_Years', 'Years_Since_Last_Review'])

X = df.drop(columns=['Termd'])
y = df['Termd']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [3]:
## --- Other Top 10 features ---
top10_colx_2 = [
    "RecruitmentSource_Google Search",
    "Department_Production       ",
    "FromDiversityJobFairID",
    "RecruitmentSource_Diversity Job Fair",
    "SpecialProjectsCount",
    "Remote_Work_Frequency",
    "DaysLateLast30",
    "MaritalDesc_Single",
    "Department_IT/IS",
    "RecruitmentSource_Indeed"
]

top10_colx_2 =  np.array(top10_colx_2)

In [4]:
model = {
    "Random Forest (Tuned Frugal Number 2)": {
        "model": RandomForestClassifier(
            n_estimators=200, 
            max_depth=8, 
            min_samples_leaf=5,
            class_weight='balanced_subsample', # Spécifique au RF
            random_state=42
        ),
        "X_train": X_train[top10_colx_2],
        "X_test": X_test[top10_colx_2],
        "scale": False
    }
}

In [6]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, roc_auc_score
import time
import pandas as pd

print("=== Frugal AI Comparison: Performance & Ethics ===\n")
# Mise à jour de l'en-tête pour inclure les nouvelles métriques
header = f"{'Model':<30} {'Feat.':>6} {'AUC':>6} {'Acc':>6} {'F1':>6} {'Prec':>6} {'Time(s)':>9} {'CO2(kg)':>10}"
print(header)
print("-" * len(header))

for name, config in model.items():
    Xtr = config["X_train"]
    Xte = config["X_test"]
    
    # Scaling si nécessaire
    if config["scale"]:
        from sklearn.preprocessing import StandardScaler
        scaler = StandardScaler()
        Xtr = pd.DataFrame(scaler.fit_transform(Xtr), columns=Xtr.columns)
        Xte = pd.DataFrame(scaler.transform(Xte), columns=Xte.columns)
    
    # Tracking des émissions et du temps
    tracker = EmissionsTracker(project_name=name, log_level="error")
    tracker.start()
    t0 = time.time()
    
    config["model"].fit(Xtr, y_train)
    
    train_time = time.time() - t0
    emissions = tracker.stop()
    
    # Calcul des prédictions
    proba = config["model"].predict_proba(Xte)[:, 1]
    preds = config["model"].predict(Xte) # Prédictions binaires (0 ou 1)
    
    # Calcul des métriques
    auc = roc_auc_score(y_test, proba)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    prec = precision_score(y_test, preds, zero_division=0)
    n_features = Xtr.shape[1]
    
    # Affichage formaté
    print(f"{name:<30} {n_features:>6} {auc:>6.2f} {acc:>6.2f} {f1:>6.2f} {prec:>6.2f} {train_time:>9.3f} {emissions:>10.6f}")


[codecarbon WARNING @ 09:36:07] Multiple instances of codecarbon are allowed to run at the same time.


=== Frugal AI Comparison: Performance & Ethics ===

Model                           Feat.    AUC    Acc     F1   Prec   Time(s)    CO2(kg)
--------------------------------------------------------------------------------------
Random Forest (Tuned Frugal Number 2)     10   0.83   0.73   0.65   0.57     1.580   0.000002
